In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise RuntimeError("Could not locate project root (no .git found above cwd)")

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments" / "concept_drift"


# Concept Drift Detection on Synthetic Artifacts (Discriminator + Deviation Features)

This notebook is a variant of `concept_drift_detection_with_syntetic_data-disc.ipynb` that adds
per-band squared-deviation features to the linear discriminator's input.

Why: the plain linear discriminator (raw/scaled S2 bands + logistic regression) cannot detect
`noise_injection` artifacts at all - test AUC sits at exactly 0.5000 across every severity level
tried. `noise_injection` adds zero-mean Gaussian noise, so it only changes the *variance* of the
perturbed year's features, never their mean. A linear classifier separates classes with a
threshold on a linear projection of the features; two classes that share the same mean and
differ only in variance are provably inseparable by any such threshold (the Bayes-optimal rule
there is non-monotonic, roughly `|x - mean|`-shaped), so no amount of severity moves the AUC.

Fix used here: after scaling, append each standardized feature's square (`z**2`) to the feature
vector, then re-scale the combined set before fitting. `E[z**2] = Var(z)`, so this new feature's
*mean* differs between classes exactly when variance differs between them - giving the linear
model something it can actually threshold on. This is applied uniformly to every method section
(mean/variance shift, covariance shift, noise injection, and the unmodified reference), not just
noise_injection, so mean-shift and covariance-shift artifacts keep their original signal plus
this new one.

This notebook keeps its own checkpoint file, separate from the original `-disc.ipynb`, so the
two can be run and compared independently.

Workflow summary:
- Load synthetic artifact manifest from `synthetic_drift_data/`
- Reuse train/val/test pixel splits from `data_split.npz`
- Run the same SGD discriminator pipeline (now with deviation features) for each artifact
- Organize execution by synthetic method type:
  - Mean/variance shift
  - Covariance shift
  - Noise injection

Each method section runs every `(mode, severity)` artifact available for that method - the
generator now sweeps a list of severity levels per method, so there are typically several
artifacts per mode, not one. Each is uniquely identified by `artifact_label` (method | mode |
severity) and results are grouped/plotted by severity to show a detection-power-vs-magnitude
curve rather than being averaged away.

A high ROC-AUC indicates stronger separability between adjacent years (drift signal).

If an artifact's metadata has `class_filter` set (concept-drift artifacts, where the
perturbation only touched pixels of one disturbance class), this notebook also reports AUC
restricted to that class's pixels (`*_auc_class_filtered`) alongside the whole-population AUC
- the whole-population number can understate a concept-drift signal that only affects a small
subset of pixels.

**Resumability:** every `(artifact, year-pair)` result is checkpointed to
`linear_disc_noise_detection_checkpoints/results.pkl` as soon as it's computed.
If the kernel is interrupted or restarted, re-running the notebook picks up where it left off -
already-computed pairs print `CACHED: ...` and are not refit. Call `clear_results_checkpoint()`
if you change the modeling pipeline and want to force a full recompute.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Paths
split_path = str(PROJECT_ROOT / "data_split.npz")
manifest_json_path = PROJECT_ROOT / "synthetic_drift_data" / "synthetic_generation_manifest.json"
manifest_csv_path = PROJECT_ROOT / "synthetic_drift_data" / "synthetic_generation_manifest.csv"
original_data_path = str(PROJECT_ROOT / "training_data_with_features.zarr")  # unmodified source, used as a reference baseline

# Checkpointing (resumability): every (artifact, year-pair) result is persisted here as soon
# as it's computed, so an interrupted/restarted run can skip already-completed pairs.
CHECKPOINT_DIR = EXPERIMENTS_DIR / "linear_disc_noise_detection_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CHECKPOINT_PATH = CHECKPOINT_DIR / "results.pkl"

# Runtime config
SAMPLE_FRACTION = 1.0  # Use 1.0 for full split, 0.1 for quick checks

# Load manifest (consume-only workflow)
if manifest_json_path.exists():
    with open(manifest_json_path, "r", encoding="utf-8") as handle:
        manifest_obj = json.load(handle)
    manifest_df = pd.DataFrame(manifest_obj.get("rows", []))
elif manifest_csv_path.exists():
    manifest_df = pd.read_csv(manifest_csv_path)
else:
    raise FileNotFoundError(
        "Synthetic manifest not found. Expected one of:\n"
        f"  - {manifest_json_path}\n"
        f"  - {manifest_csv_path}\n"
        "Generate artifacts first in 'data drift testing on syntetic data.ipynb'."
    )

required_cols = {"mode", "method", "zarr_path", "metadata_path"}
missing_cols = sorted(required_cols - set(manifest_df.columns))
if missing_cols:
    raise ValueError(f"Manifest is missing required columns: {missing_cols}")

if manifest_df.empty:
    raise ValueError("Manifest contains no artifact rows.")

manifest_df = manifest_df.copy()

# Older manifests (pre severity-sweep) won't have these columns; default them so the rest of
# this notebook can treat every manifest uniformly.
for optional_col, default_value in [("severity", None), ("pixel_fraction", 1.0), ("class_filter", None)]:
    if optional_col not in manifest_df.columns:
        manifest_df[optional_col] = default_value


def _severity_display(severity):
    """Render a severity value (dict for mean_variance_shift, scalar for others) as a short label."""
    if isinstance(severity, dict):
        return ", ".join(f"{k}={v}" for k, v in severity.items())
    if severity is None or (isinstance(severity, float) and np.isnan(severity)):
        return "n/a"
    return str(severity)


manifest_df["severity_display"] = manifest_df["severity"].apply(_severity_display)

# One artifact = one (method, mode, severity) combination now that the generator sweeps a
# severity list per method; the label must include severity or rows silently collide.
manifest_df["artifact_label"] = (
    manifest_df["method"] + " | " + manifest_df["mode"] + " | " + manifest_df["severity_display"]
)
manifest_df = manifest_df.sort_values(["method", "mode"]).reset_index(drop=True)

# Load data splits
splits = np.load(split_path)
train_pixel_indices = splits["train_pixel_indices"]
val_pixel_indices = splits["val_pixel_indices"]
test_pixel_indices = splits["test_pixel_indices"]

print(f"Loaded manifest rows: {len(manifest_df)}")
print(f"Split sizes: train={len(train_pixel_indices):,}, val={len(val_pixel_indices):,}, test={len(test_pixel_indices):,}")
display(manifest_df[["method", "mode", "severity_display", "pixel_fraction", "class_filter", "zarr_path", "metadata_path"]])

In [ ]:
# Manifest diagnostics
method_order = ["mean_variance_shift", "covariance_shift", "noise_injection"]
mode_order = ["baseline_replication", "per_year_original"]

summary_df = (
    manifest_df.groupby(["method", "mode"], dropna=False)
    .size()
    .rename("n_artifacts")
    .reset_index()
    .sort_values(["method", "mode"])
    .reset_index(drop=True)
)

display(summary_df)

detail_df = (
    manifest_df.groupby(["method", "mode", "severity_display"], dropna=False)
    .size()
    .rename("n_artifacts")
    .reset_index()
    .sort_values(["method", "mode", "severity_display"])
    .reset_index(drop=True)
)

display(detail_df)

def _resolve_manifest_path(p):
    p = Path(p)
    return p if p.is_absolute() else PROJECT_ROOT / p

missing_files = []
for _, row in manifest_df.iterrows():
    zarr_ok = _resolve_manifest_path(row["zarr_path"]).exists()
    meta_ok = _resolve_manifest_path(row["metadata_path"]).exists()
    if not (zarr_ok and meta_ok):
        missing_files.append({
            "method": row["method"],
            "mode": row["mode"],
            "severity_display": row["severity_display"],
            "zarr_exists": zarr_ok,
            "metadata_exists": meta_ok,
        })

if missing_files:
    missing_df = pd.DataFrame(missing_files)
    display(missing_df)
    raise FileNotFoundError("Some artifact files from manifest are missing.")

print("All manifest-referenced artifact files are present.")

In [ ]:
RESULTS_CHECKPOINT_COLUMNS = [
    "method", "mode", "severity", "severity_display", "pixel_fraction", "class_filter",
    "artifact_label", "zarr_path", "pair", "year_t_minus_1", "year_t", "sample_fraction",
    "n_train", "n_val", "n_test", "val_auc", "test_auc", "val_auc_class_filtered",
    "test_auc_class_filtered", "val_f1", "test_f1", "time_sec", "changed_years",
    "perturb_year_values",
]


def load_results_checkpoint(path=RESULTS_CHECKPOINT_PATH):
    """Load the on-disk results checkpoint, or an empty frame with the expected schema."""
    if path.exists():
        return pd.read_pickle(path)
    return pd.DataFrame(columns=RESULTS_CHECKPOINT_COLUMNS)


def save_results_checkpoint(df, path=RESULTS_CHECKPOINT_PATH):
    """Write the results checkpoint atomically (tmp file + replace) so a crash mid-write can't corrupt it."""
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df.to_pickle(tmp_path)
    tmp_path.replace(path)


def clear_results_checkpoint(path=RESULTS_CHECKPOINT_PATH):
    """Delete the on-disk checkpoint and reset the in-memory cache. Call this after changing the modeling pipeline."""
    global CHECKPOINT_DF
    if path.exists():
        path.unlink()
    CHECKPOINT_DF = pd.DataFrame(columns=RESULTS_CHECKPOINT_COLUMNS)
    print(f"Cleared results checkpoint at {path}")


def prepare_features_for_year(ds_subset, year_idx):
    """Prepare per-pixel features for a given year index using S2 bands only."""
    if year_idx == 0:
        return None, None

    valid_mask = ds_subset.disturbances[:, year_idx].values != 255

    s2_data = ds_subset.s2_bands.values  # shape (n_pixels, n_years, n_bands)
    s2_t = s2_data[:, year_idx, :].copy()  # shape (n_pixels, n_bands)

    # Vectorized NaN imputation: per-pixel, per-band mean across years
    nan_mask = np.isnan(s2_t)
    if np.any(nan_mask):
        band_means = np.nanmean(s2_data, axis=1)  # shape (n_pixels, n_bands)
        s2_t[nan_mask] = band_means[nan_mask]

    X = s2_t
    y = ds_subset.disturbances[:, year_idx].values

    # Combined mask: valid disturbance + binary label + finite features
    keep_mask = valid_mask & np.isin(y, [0, 1]) & np.all(np.isfinite(X), axis=1)
    X = X[keep_mask]
    y = y[keep_mask]

    if len(X) == 0:
        return None, None

    return X, y


def build_pair_dataset_cached(X_prev, y_prev, X_curr, y_curr):
    """Build binary year-pair dataset from pre-computed features.

    Returns (X_pair, pair_label, disturbance_label). pair_label is 0/1 for which year a row
    came from - the discriminator's actual training target. disturbance_label is the original
    0/1 disturbance class for that row, kept alongside so class-conditional AUC can be
    computed for concept-drift artifacts (class_filter != None) without retraining: the
    whole-population AUC can understate a shift that only touched one class's pixels.
    """
    if X_prev is None or X_curr is None:
        return None, None, None

    if X_prev.shape[1] != X_curr.shape[1]:
        return None, None, None

    X_pair = np.vstack([X_prev, X_curr])
    pair_label = np.concatenate([
        np.zeros(len(y_prev), dtype=np.uint8),
        np.ones(len(y_curr), dtype=np.uint8),
    ])
    disturbance_label = np.concatenate([y_prev, y_curr]).astype(np.uint8)

    return X_pair, pair_label, disturbance_label


def add_deviation_features(X_scaled):
    """Append per-band squared-deviation features to an already-standardized matrix.

    A pure variance shift (e.g. noise_injection's zero-mean noise) leaves the mean of the
    scaled features unchanged, so a linear boundary on them alone sits at chance regardless
    of severity. z**2 has mean = Var(z), which does differ between classes when variance
    differs, giving the linear model a feature it can actually threshold on.
    """
    return np.hstack([X_scaled, X_scaled ** 2])


def _sample_indices(indices, sample_fraction, rng_seed=RANDOM_STATE):
    if sample_fraction >= 1.0:
        return indices
    rng_local = np.random.default_rng(rng_seed)
    sample_size = max(1, int(len(indices) * sample_fraction))
    return rng_local.choice(indices, size=sample_size, replace=False)


def _class_conditional_auc(y_true, y_proba, disturbance_label, target_class):
    """AUC restricted to rows whose original disturbance label equals target_class.

    Returns NaN when there is no class filter, or when the restricted subset doesn't have
    both pair_label classes present (AUC is undefined there).
    """
    if target_class is None or disturbance_label is None:
        return np.nan
    mask = disturbance_label == target_class
    if mask.sum() < 2 or len(np.unique(y_true[mask])) < 2:
        return np.nan
    return float(roc_auc_score(y_true[mask], y_proba[mask]))


def to_auc_display(df, id_cols):
    """Trim a results dataframe down to identifying columns plus a single 'AUC' column (test AUC).

    Every table in this notebook displays test AUC only, under the name 'AUC' - val AUC,
    class-filtered AUC, and F1 stay in the underlying results_df for plotting but aren't
    shown in tables.
    """
    cols = [c for c in id_cols if c in df.columns] + ["test_auc"]
    return df[cols].rename(columns={"test_auc": "AUC"})


def plot_artifact_auc(results_df, title_suffix="", reference_auc=None):
    """Plot test AUC ('AUC') per year pair.

    reference_auc, when given, is a sequence aligned to results_df's rows - the
    original-unmodified-data AUC for the same year pairs - drawn as a dashed reference line
    so a per_year_original artifact's perturbed-pair results can be read against it.
    """
    if len(results_df) == 0:
        print("No rows to plot for this artifact.")
        return

    x = np.arange(len(results_df))
    plt.figure(figsize=(10, 4.5))
    plt.plot(x, results_df["test_auc"], marker="o", label="AUC")

    if reference_auc is not None:
        plt.plot(x, reference_auc, marker="d", linestyle="--", color="gray", label="AUC (original, unmodified)")

    plt.xticks(x, results_df["pair"], rotation=45, ha="right")
    plt.ylim(0.45, 1.0)
    plt.ylabel("ROC-AUC")
    plt.title(f"Adjacent-Year Drift Discriminator AUC{title_suffix}")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def run_discriminator_on_dataset(
    zarr_path,
    method,
    mode,
    artifact_label,
    severity=None,
    severity_display=None,
    pixel_fraction=None,
    class_filter=None,
    changed_years=None,
    perturb_year_values=None,
    sample_fraction=SAMPLE_FRACTION,
):
    """Run the adjacent-year discriminator pipeline against any zarr dataset.

    Shared by synthetic artifacts (via run_discriminator_for_artifact) and the unmodified
    original dataset (the per_year_original reference) so both paths train/evaluate exactly
    the same way.

    Resumability: each (artifact_label, pair, sample_fraction) result is looked up in the
    on-disk checkpoint (CHECKPOINT_DF) before doing any work for that pair. A hit skips
    feature prep and model fitting entirely; a miss computes the pair as before, then appends
    the new row to the checkpoint and saves it to disk immediately.
    """
    global CHECKPOINT_DF

    zarr_path = Path(zarr_path)
    if not zarr_path.is_absolute():
        zarr_path = PROJECT_ROOT / zarr_path

    ds_artifact = xr.open_zarr(zarr_path)
    year_values = ds_artifact.year.values
    n_years = len(year_values)
    n_pairs = n_years - 2

    train_sample = _sample_indices(train_pixel_indices, sample_fraction, rng_seed=RANDOM_STATE)
    val_sample = _sample_indices(val_pixel_indices, sample_fraction, rng_seed=RANDOM_STATE + 1)
    test_sample = _sample_indices(test_pixel_indices, sample_fraction, rng_seed=RANDOM_STATE + 2)

    ds_train = ds_artifact.isel(pixel=train_sample)
    ds_val = ds_artifact.isel(pixel=val_sample)
    ds_test = ds_artifact.isel(pixel=test_sample)

    print(f"Artifact: {artifact_label}")
    print(f"Dataset: {zarr_path}")
    print(f"Pairs to evaluate: {n_pairs}")

    results = []
    skipped_pairs = []
    start_time = time.time()

    for pair_idx, idx_curr in enumerate(range(2, n_years), start=1):
        idx_prev = idx_curr - 1
        year_prev = int(year_values[idx_prev])
        year_curr = int(year_values[idx_curr])
        pair_name = f"{year_prev} vs {year_curr}"

        pair_start = time.time()
        print(f"\n[{pair_idx}/{n_pairs}] {pair_name}")

        cache_hit = CHECKPOINT_DF[
            (CHECKPOINT_DF["artifact_label"] == artifact_label)
            & (CHECKPOINT_DF["pair"] == pair_name)
            & (CHECKPOINT_DF["sample_fraction"] == sample_fraction)
        ]
        if len(cache_hit) > 0:
            cached_row = cache_hit.iloc[-1].to_dict()
            results.append(cached_row)
            print(
                f"  CACHED: Val AUC = {cached_row['val_auc']:.4f} | Test AUC = {cached_row['test_auc']:.4f} "
                f"| Val F1 = {cached_row['val_f1']:.4f} | Test F1 = {cached_row['test_f1']:.4f}"
            )
            continue

        X_train_prev, y_train_prev = prepare_features_for_year(ds_train, idx_prev)
        X_train_curr, y_train_curr = prepare_features_for_year(ds_train, idx_curr)
        X_train, y_train, _ = build_pair_dataset_cached(X_train_prev, y_train_prev, X_train_curr, y_train_curr)

        X_val_prev, y_val_prev = prepare_features_for_year(ds_val, idx_prev)
        X_val_curr, y_val_curr = prepare_features_for_year(ds_val, idx_curr)
        X_val, y_val, disturbance_val = build_pair_dataset_cached(X_val_prev, y_val_prev, X_val_curr, y_val_curr)

        X_test_prev, y_test_prev = prepare_features_for_year(ds_test, idx_prev)
        X_test_curr, y_test_curr = prepare_features_for_year(ds_test, idx_curr)
        X_test, y_test, disturbance_test = build_pair_dataset_cached(X_test_prev, y_test_prev, X_test_curr, y_test_curr)

        if any(v is None for v in [X_train, y_train, X_val, y_val, X_test, y_test]):
            skipped_pairs.append((pair_name, "missing or invalid data after filtering"))
            print("  SKIP: missing data")
            continue

        if len(np.unique(y_train)) < 2 or len(np.unique(y_val)) < 2 or len(np.unique(y_test)) < 2:
            skipped_pairs.append((pair_name, "single class present in splits"))
            print("  SKIP: single class in one split")
            continue

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        X_test_scaled = scaler.transform(X_test)

        feat_scaler = StandardScaler()
        X_train_feat = feat_scaler.fit_transform(add_deviation_features(X_train_scaled))
        X_val_feat = feat_scaler.transform(add_deviation_features(X_val_scaled))
        X_test_feat = feat_scaler.transform(add_deviation_features(X_test_scaled))

        model = SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=0.0001,
            max_iter=1000,
            tol=1e-3,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        model.fit(X_train_feat, y_train)

        val_proba = model.predict_proba(X_val_feat)[:, 1]
        test_proba = model.predict_proba(X_test_feat)[:, 1]

        val_auc = roc_auc_score(y_val, val_proba)
        test_auc = roc_auc_score(y_test, test_proba)
        val_pred = (val_proba >= 0.5).astype(int)
        test_pred = (test_proba >= 0.5).astype(int)
        val_f1 = f1_score(y_val, val_pred)
        test_f1 = f1_score(y_test, test_pred)

        val_auc_class_filtered = _class_conditional_auc(y_val, val_proba, disturbance_val, class_filter)
        test_auc_class_filtered = _class_conditional_auc(y_test, test_proba, disturbance_test, class_filter)

        pair_time = time.time() - pair_start

        row = {
            "method": method,
            "mode": mode,
            "severity": severity,
            "severity_display": severity_display,
            "pixel_fraction": pixel_fraction,
            "class_filter": class_filter,
            "artifact_label": artifact_label,
            "zarr_path": str(zarr_path),
            "pair": pair_name,
            "year_t_minus_1": year_prev,
            "year_t": year_curr,
            "sample_fraction": sample_fraction,
            "n_train": int(len(y_train)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "val_auc": float(val_auc),
            "test_auc": float(test_auc),
            "val_auc_class_filtered": val_auc_class_filtered,
            "test_auc_class_filtered": test_auc_class_filtered,
            "val_f1": float(val_f1),
            "test_f1": float(test_f1),
            "time_sec": float(pair_time),
            "changed_years": changed_years,
            "perturb_year_values": perturb_year_values,
        }
        results.append(row)

        CHECKPOINT_DF = pd.concat([CHECKPOINT_DF, pd.DataFrame([row])], ignore_index=True)
        save_results_checkpoint(CHECKPOINT_DF)

        auc_msg = (
            f"  RESULTS: Val AUC = {val_auc:.4f} | Test AUC = {test_auc:.4f} "
            f"| Val F1 = {val_f1:.4f} | Test F1 = {test_f1:.4f}"
        )
        if class_filter is not None:
            auc_msg += (
                f" | Val AUC[class={class_filter}] = {val_auc_class_filtered:.4f} "
                f"| Test AUC[class={class_filter}] = {test_auc_class_filtered:.4f}"
            )
        print(auc_msg)

    total_time = time.time() - start_time
    results_df = pd.DataFrame(results).sort_values("year_t").reset_index(drop=True)

    print("\n" + "=" * 70)
    print(f"COMPLETE: {artifact_label}")
    print(f"Elapsed: {total_time:.1f}s ({total_time/60:.1f}m)")
    if len(results_df) > 0:
        print(f"Pairs trained: {len(results_df)}")
        print(f"Average time per pair: {results_df['time_sec'].mean():.1f}s")
        print(f"Min/Max AUC (test): {results_df['test_auc'].min():.3f} / {results_df['test_auc'].max():.3f}")
    else:
        print("No valid year pairs were found.")
    print("=" * 70 + "\n")

    return results_df, skipped_pairs


def run_discriminator_for_artifact(artifact_row, sample_fraction=SAMPLE_FRACTION):
    """Run the adjacent-year discriminator pipeline for a single synthetic artifact row."""
    artifact_row = artifact_row.copy()

    metadata_path = Path(artifact_row["metadata_path"])
    if not metadata_path.is_absolute():
        metadata_path = PROJECT_ROOT / metadata_path
    with open(metadata_path, "r", encoding="utf-8") as handle:
        metadata = json.load(handle)

    return run_discriminator_on_dataset(
        zarr_path=artifact_row["zarr_path"],
        method=str(artifact_row["method"]),
        mode=str(artifact_row["mode"]),
        artifact_label=str(artifact_row["artifact_label"]),
        severity=metadata.get("severity"),
        severity_display=artifact_row.get("severity_display"),
        pixel_fraction=metadata.get("pixel_fraction"),
        class_filter=metadata.get("class_filter"),
        changed_years=metadata.get("changed_years"),
        perturb_year_values=metadata.get("perturb_year_values"),
        sample_fraction=sample_fraction,
    )


def run_method_section(method_name, sample_fraction=SAMPLE_FRACTION):
    """Run every (mode, severity) artifact for one synthetic method and return section outputs."""
    section_rows = manifest_df[manifest_df["method"] == method_name].copy()
    if section_rows.empty:
        print(f"No artifacts found for method: {method_name}")
        return {"method": method_name, "artifacts": {}, "combined": pd.DataFrame()}

    mode_rank = {"baseline_replication": 0, "per_year_original": 1}
    section_rows["_mode_rank"] = section_rows["mode"].map(mode_rank).fillna(99)
    # Stable sort preserves each mode's original severity-sweep order from the manifest.
    section_rows = section_rows.sort_values(["_mode_rank", "mode"], kind="stable").reset_index(drop=True)

    artifact_outputs = {}
    combined_frames = []

    for _, row in section_rows.iterrows():
        artifact_label = row["artifact_label"]
        print(f"\n### Running artifact: {artifact_label}")
        results_df, skipped_pairs = run_discriminator_for_artifact(row, sample_fraction=sample_fraction)

        # Keyed by artifact_label (mode + severity), not mode alone: with the severity sweep
        # there are now multiple rows per mode, and keying by mode alone would silently
        # overwrite all but the last severity level's results.
        artifact_outputs[artifact_label] = {
            "results_df": results_df,
            "skipped_pairs": skipped_pairs,
            "row": row.to_dict(),
        }

        if len(results_df) > 0:
            display_df = to_auc_display(results_df, ["pair", "year_t_minus_1", "year_t"])
            reference_auc = None
            if row["mode"] == "per_year_original" and original_reference_df is not None:
                # Reference is joined per year pair (not a single scalar) since natural
                # year-to-year AUC can vary pair to pair even with no perturbation at all.
                reference_cols = original_reference_df[["year_t_minus_1", "year_t", "AUC"]].rename(
                    columns={"AUC": "AUC (original, unmodified)"}
                )
                display_df = display_df.merge(reference_cols, on=["year_t_minus_1", "year_t"], how="left")
                reference_auc = display_df["AUC (original, unmodified)"]
            display(display_df)
            plot_artifact_auc(results_df, title_suffix=f" | {artifact_label}", reference_auc=reference_auc)
            combined_frames.append(results_df)

        if skipped_pairs:
            print("Skipped pairs:")
            for pair_name, reason in skipped_pairs:
                print(f"  - {pair_name}: {reason}")

    if combined_frames:
        combined = pd.concat(combined_frames, ignore_index=True)
    else:
        combined = pd.DataFrame()

    return {
        "method": method_name,
        "artifacts": artifact_outputs,
        "combined": combined,
    }


def summarize_and_plot_section(combined_df, method_name):
    """Severity-aware summary table + detection-power-vs-severity plot, shared by every method section."""
    if combined_df.empty:
        print(f"No results to summarize for {method_name}.")
        return pd.DataFrame()

    summary = (
        combined_df.groupby(["mode", "severity_display"], as_index=False, sort=False)
        .agg(
            mean_test_auc=("test_auc", "mean"),
            n_pairs=("pair", "count"),
        )
    )
    print(f"{method_name} combined view:")
    display_summary = summary.rename(columns={"mean_test_auc": "AUC"})
    is_pyo = display_summary["mode"] == "per_year_original"
    if original_reference_mean_auc is not None and is_pyo.any():
        display_summary.loc[is_pyo, "AUC (original, unmodified)"] = original_reference_mean_auc
    display(display_summary)

    plt.figure(figsize=(10, 4.5))
    for mode, mode_df in summary.groupby("mode", sort=False):
        plt.plot(mode_df["severity_display"], mode_df["mean_test_auc"], marker="o", label=f"{mode} (AUC)")
    if original_reference_mean_auc is not None and (summary["mode"] == "per_year_original").any():
        plt.axhline(
            original_reference_mean_auc, color="gray", linestyle="--",
            label="AUC (original, unmodified)",
        )
    plt.ylabel("Mean AUC across year pairs")
    plt.xlabel("Severity level")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0.45, 1.0)
    plt.title(f"{method_name}: detection power vs. severity")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return summary

In [ ]:
# Global container for section outputs
section_results = {}
original_reference_df = None
original_reference_mean_auc = None

# Resumability: load any previously-computed (artifact, pair) results from disk. Cells below
# will skip recomputing anything already present here.
CHECKPOINT_DF = load_results_checkpoint()
print(f"Loaded {len(CHECKPOINT_DF)} cached (artifact, pair) result(s) from {RESULTS_CHECKPOINT_PATH}")

print("Ready. Run the original-data reference cell, then the method sections below.")

## Original (Unmodified) Reference

Run the same adjacent-year discriminator directly on `training_data_with_features.zarr`,
with no synthetic perturbation applied at all. `per_year_original` mode keeps every year's
real data and only perturbs two of them, so most of its year pairs already reflect natural
year-to-year variation rather than an injected shift. This reference AUC (per pair, and its
mean) is joined into every `per_year_original` table below so perturbed results can be read
against the unperturbed baseline instead of in isolation.

In [ ]:
original_results_df, original_skipped_pairs = run_discriminator_on_dataset(
    zarr_path=original_data_path,
    method="none",
    mode="original_unmodified",
    artifact_label="original_unmodified",
    sample_fraction=SAMPLE_FRACTION,
)

original_reference_df = to_auc_display(original_results_df, ["pair", "year_t_minus_1", "year_t"])
original_reference_mean_auc = (
    float(original_reference_df["AUC"].mean()) if len(original_reference_df) > 0 else None
)

print("Original (unmodified) reference AUC per year pair:")
display(original_reference_df)
if original_reference_mean_auc is not None:
    print(f"Mean reference AUC across pairs: {original_reference_mean_auc:.4f}")

## Mean/Variance Shift

Run discriminator on synthetic artifacts generated with `mean_variance_shift` for both modes.

In [ ]:
print("=== Mean/Variance Shift ===")
print("Running discriminator for mean_variance_shift artifacts...")

In [ ]:
section_results["mean_variance_shift"] = run_method_section(
    method_name="mean_variance_shift",
    sample_fraction=SAMPLE_FRACTION,
 )

mean_combined = section_results["mean_variance_shift"]["combined"]
mean_summary = summarize_and_plot_section(mean_combined, "mean_variance_shift")

## Covariance Shift

Run discriminator on synthetic artifacts generated with `covariance_shift` for both modes.

In [ ]:
section_results["covariance_shift"] = run_method_section(
    method_name="covariance_shift",
    sample_fraction=SAMPLE_FRACTION,
 )

cov_combined = section_results["covariance_shift"]["combined"]
cov_summary = summarize_and_plot_section(cov_combined, "covariance_shift")

## Noise Injection

Run discriminator on synthetic artifacts generated with `noise_injection` for both modes.

In [ ]:
section_results["noise_injection"] = run_method_section(
    method_name="noise_injection",
    sample_fraction=SAMPLE_FRACTION,
 )

noise_combined = section_results["noise_injection"]["combined"]
noise_summary = summarize_and_plot_section(noise_combined, "noise_injection")

## Cross-Method Summary

Consolidate all method runs into one comparison table and aggregate chart.

In [ ]:
available = {
    k: v for k, v in section_results.items()
    if isinstance(v, dict) and not v.get("combined", pd.DataFrame()).empty
}

if not available:
    print("No method results available yet. Run the method section cells first.")
else:
    all_results = pd.concat([v["combined"] for v in available.values()], ignore_index=True)
    print(f"Collected rows: {len(all_results)}")
    display(
        to_auc_display(
            all_results, ["method", "mode", "severity_display", "pair", "year_t_minus_1", "year_t"]
        ).head()
    )

    summary = (
        all_results.groupby(["method", "mode", "severity_display"], as_index=False, sort=False)
        .agg(
            mean_test_auc=("test_auc", "mean"),
            n_pairs=("pair", "count"),
        )
        .sort_values(["method", "mode"], kind="stable")
        .reset_index(drop=True)
    )

    print("Method/mode/severity summary:")
    plot_df = summary.rename(columns={"mean_test_auc": "AUC"})
    is_pyo = plot_df["mode"] == "per_year_original"
    if original_reference_mean_auc is not None and is_pyo.any():
        plot_df.loc[is_pyo, "AUC (original, unmodified)"] = original_reference_mean_auc
    display(plot_df)

    # x-axis is one bar per (method, mode, severity) artifact - averaging away severity here
    # (as the original method-only chart did) would hide the whole point of the sweep.
    plot_df["artifact_label"] = (
        plot_df["method"] + " | " + plot_df["mode"] + " | " + plot_df["severity_display"]
    )
    bar_cols = [c for c in ["AUC", "AUC (original, unmodified)"] if c in plot_df.columns]
    ax = plot_df.plot(
        x="artifact_label",
        y=bar_cols,
        kind="bar",
        figsize=(14, 5),
        rot=75,
        title="Average AUC by Synthetic Artifact (method | mode | severity)",
    )
    ax.set_xlabel("")
    ax.set_ylabel("ROC-AUC")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()